# 02 - EDA Business & Storytelling (Slide-ready)

Ce notebook repond a des questions business clefs pour JO 2028.
Chaque section contient:
- une question business
- un graphique exploitable en soutenance
- un angle narratif YPerf


## Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from pathlib import Path

sns.set_theme(style="darkgrid")
pd.set_option("display.max_columns", 60)

DATA_PATH = Path("../data/raw/olympics_dataset.csv")
assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"

raw = pd.read_csv(DATA_PATH, low_memory=False)
# Ecarter les lignes brutes decalees (Year non numerique) avant toute analyse temporelle.
raw = raw[pd.to_numeric(raw["Year"], errors="coerce").notna()].copy()
raw["Year"] = raw["Year"].astype(int)
raw["Medal"] = raw["Medal"].fillna("None")
raw["is_medal"] = (raw["Medal"] != "None").astype(int)
raw["medal_points"] = raw["Medal"].map({"Gold": 3, "Silver": 2, "Bronze": 1}).fillna(0).astype(int)

raw.head(2)


,player_id,Name,Sex,Team,NOC,Year,Season,City,Sport,Event,Medal,Unnamed: 11,is_medal,medal_points
0,0,A Dijiang,M,China,CHN,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,No medal,NaN,1,0
1,1,A Lamusi,M,China,CHN,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,No medal,NaN,1,0


## Q1. Quelle est la dynamique globale de la competition olympique?
**Narratif:** le niveau d'intensite (participations + medailles) evolue dans le temps, ce qui impacte la difficulte de prediction.


In [2]:
year_kpis = (
    raw.groupby("Year", as_index=False)
    .agg(entries=("player_id", "count"), medals=("is_medal", "sum"), countries=("NOC", "nunique"))
    .sort_values("Year")
)

fig = px.line(year_kpis, x="Year", y=["entries", "medals"], markers=True,
              title="Global olympic activity over time")
fig.show()
year_kpis.tail(10)


,Year,entries,medals,countries
21,1988,12037,12037,159
22,1992,12977,12977,169
23,1996,13780,13780,197
24,2000,13821,13821,200
25,2004,13443,13443,201
26,2008,13602,13602,204
27,2012,12920,12920,205
28,2016,13688,13688,207
29,2020,15120,15120,206
30,2024,14892,14892,206


## Q2. Quels pays dominent historiquement les medailles?
**Narratif:** identifier les acteurs structurellement forts (base line-up des favoris 2028).


In [3]:
country_medals = (
    raw.groupby("NOC", as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
    .sort_values("medals", ascending=False)
)

fig = px.bar(country_medals.head(15), x="NOC", y="medals", title="Top 15 countries by total medals")
fig.show()
country_medals.head(15)


,NOC,medals
220,USA,16774
76,GBR,11998
71,FRA,11971
102,ITA,9351
81,GER,8866
13,AUS,8379
37,CAN,7907
106,JPN,7721
92,HUN,6621
197,SWE,6422


## Q3. Quels pays sont en acceleration recente (momentum)?
**Narratif:** differencier les leaders historiques des nations en progression depuis 2016.


In [4]:
recent_start = 2016

country_year = (
    raw.groupby(["NOC", "Year"], as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
)

hist = country_year[country_year["Year"] < recent_start].groupby("NOC", as_index=False)["medals"].mean().rename(columns={"medals": "hist_avg"})
recent = country_year[country_year["Year"] >= recent_start].groupby("NOC", as_index=False)["medals"].mean().rename(columns={"medals": "recent_avg"})

momentum = hist.merge(recent, on="NOC", how="outer").fillna(0)
momentum["delta"] = momentum["recent_avg"] - momentum["hist_avg"]
momentum = momentum[momentum["recent_avg"] >= 5].sort_values("delta", ascending=False)

fig = px.bar(momentum.head(15), x="NOC", y="delta", title="Countries with strongest recent medal momentum")
fig.show()
momentum.head(15)


,NOC,hist_avg,recent_avg,delta
171,ROC,0.000000,561.000000,561.000000
13,AUS,252.846154,601.666667,348.820513
106,JPN,280.952381,607.000000,326.047619
30,BRA,145.500000,466.666667,321.166667
220,USA,531.296296,809.666667,278.370370
71,FRA,361.464286,616.666667,255.202381
37,CAN,248.269231,484.000000,235.730769
42,CHN,318.666667,553.333333,234.666667
65,ESP,206.545455,439.333333,232.787879
102,ITA,279.214286,511.000000,231.785714


## Q4. Quels sports concentrent la creation de medailles?
**Narratif:** prioriser les sports a fort volume pour maximiser l'impact de prediction.


In [5]:
sport_medals = (
    raw.groupby("Sport", as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
    .sort_values("medals", ascending=False)
)

fig = px.bar(sport_medals.head(15), x="Sport", y="medals", title="Top sports by medal volume")
fig.update_layout(xaxis_tickangle=-35)
fig.show()
sport_medals.head(15)


,Sport,medals
8,Athletics,43294
38,Gymnastics,26707
63,Swimming,26416
58,Shooting,12580
54,Rowing,11625
34,Fencing,11558
22,Cycling,10859
36,Football,7906
75,Wrestling,7734
57,Sailing,7266


## Q5. Quels sports accelerent recemment?
**Narratif:** detecter les disciplines dont le poids recent augmente.


In [6]:
sport_year = (
    raw.groupby(["Sport", "Year"], as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
)

sport_hist = sport_year[sport_year["Year"] < recent_start].groupby("Sport", as_index=False)["medals"].mean().rename(columns={"medals": "hist_avg"})
sport_recent = sport_year[sport_year["Year"] >= recent_start].groupby("Sport", as_index=False)["medals"].mean().rename(columns={"medals": "recent_avg"})

sport_momentum = sport_hist.merge(sport_recent, on="Sport", how="outer").fillna(0)
sport_momentum["delta"] = sport_momentum["recent_avg"] - sport_momentum["hist_avg"]
sport_momentum = sport_momentum.sort_values("delta", ascending=False)

fig = px.bar(sport_momentum.head(15), x="Sport", y="delta", title="Sports with strongest recent acceleration")
fig.update_layout(xaxis_tickangle=-35)
fig.show()
sport_momentum.head(15)


,Sport,hist_avg,recent_avg,delta
6,Artistic Gymnastics,0.000000,1148.500000,1148.500000
8,Athletics,1289.857143,2392.666667,1102.809524
63,Swimming,772.392857,1596.333333,823.940476
30,Cycling Track,0.000000,439.000000,439.000000
32,Equestrian,0.000000,417.500000,417.500000
18,Canoe Sprint,0.000000,378.000000,378.000000
56,Rugby Sevens,0.000000,309.333333,309.333333
36,Football,241.230769,544.666667,303.435897
22,Cycling,364.000000,667.000000,303.000000
26,Cycling Road,0.000000,246.500000,246.500000


## Q6. Quelle est la repartition du niveau de performance (Gold/Silver/Bronze)?
**Narratif:** la structure des podiums informe la robustesse competitive.


In [7]:
medal_mix = (
    raw[raw["Medal"] != "None"]
    .groupby("Medal", as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

fig = px.pie(medal_mix, names="Medal", values="count", title="Global medal composition")
fig.show()
medal_mix


,Medal,count
0,Bronze,13070
1,Gold,13002
2,No medal,213746
3,Silver,12746


## Q7. Comment evolue la participation par genre?
**Narratif:** la convergence F/M est un signal de transformation structurelle.


In [8]:
sex_year = (
    raw.groupby(["Year", "Sex"], as_index=False)
    .size()
    .rename(columns={"size": "entries"})
)

total_year = sex_year.groupby("Year", as_index=False)["entries"].sum().rename(columns={"entries": "total"})
sex_year = sex_year.merge(total_year, on="Year", how="left")
sex_year["share"] = sex_year["entries"] / sex_year["total"]

fig = px.line(sex_year, x="Year", y="share", color="Sex", markers=True,
              title="Gender participation share over time")
fig.show()
sex_year.tail(8)


,Year,Sex,entries,total,share
53,2012,F,5815,12920,0.450077
54,2012,M,7105,12920,0.549923
55,2016,F,6223,13688,0.454632
56,2016,M,7465,13688,0.545368
57,2020,F,7266,15120,0.480556
58,2020,M,7854,15120,0.519444
59,2024,F,7312,14892,0.491002
60,2024,M,7580,14892,0.508998


## Q8. Quels pays performent le mieux en efficience (medailles / participations)?
**Narratif:** detecter les pays qui convertissent le mieux leurs engagements en podiums.


In [9]:
country_eff = (
    raw.groupby("NOC", as_index=False)
    .agg(entries=("player_id", "count"), medals=("is_medal", "sum"))
)
country_eff["medal_rate"] = country_eff["medals"] / country_eff["entries"]
country_eff = country_eff[country_eff["entries"] >= 200].sort_values("medal_rate", ascending=False)

fig = px.bar(country_eff.head(15), x="NOC", y="medal_rate", title="Top countries by medal conversion rate (min 200 entries)")
fig.show()
country_eff.head(15)


,NOC,entries,medals,medal_rate
4,ALG,639,639,1.0
169,QAT,222,222,1.0
167,PRK,700,700,1.0
166,POR,1709,1709,1.0
165,POL,5459,5459,1.0
161,PHI,757,757,1.0
160,PER,596,596,1.0
157,PAK,580,580,1.0
155,NZL,2674,2674,1.0
153,NOR,2838,2838,1.0


## Q9. Quels pays sont specialises (dependance sport)?
**Narratif:** un portefeuille trop concentre augmente le risque en 2028.


In [10]:
country_sport = (
    raw.groupby(["NOC", "Sport"], as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
)

country_total = country_sport.groupby("NOC", as_index=False)["medals"].sum().rename(columns={"medals": "total_medals"})
country_sport = country_sport.merge(country_total, on="NOC", how="left")
country_sport["share"] = country_sport["medals"] / country_sport["total_medals"]

specialization = (
    country_sport[country_sport["total_medals"] >= 50]
    .sort_values(["NOC", "share"], ascending=[True, False])
    .groupby("NOC", as_index=False)
    .first()[["NOC", "Sport", "share", "total_medals"]]
    .sort_values("share", ascending=False)
)

fig = px.bar(specialization.head(15), x="NOC", y="share", color="Sport",
             title="Countries with highest sport concentration (top sport share)")
fig.show()
specialization.head(15)


,NOC,Sport,share,total_medals
103,LBR,Athletics,0.954545,88
91,JAM,Athletics,0.849746,985
54,ERI,Athletics,0.840580,69
156,SLE,Athletics,0.827869,122
65,GAM,Athletics,0.826087,69
26,BOT,Athletics,0.816000,125
58,ETH,Athletics,0.797327,449
104,LES,Athletics,0.774648,71
17,BDI,Athletics,0.722222,54
49,DJI,Athletics,0.700000,50


## Q10. Qui suivre pour 2028? (shortlist storytelling)
**Narratif:** combiner volume + momentum + efficience pour prioriser les nations a suivre.


In [11]:
shortlist = (
    country_medals.rename(columns={"medals": "total_medals"})
    .merge(momentum[["NOC", "delta", "recent_avg"]], on="NOC", how="left")
    .merge(country_eff[["NOC", "medal_rate"]], on="NOC", how="left")
    .fillna({"delta": 0, "recent_avg": 0, "medal_rate": 0})
)

shortlist["score_2028"] = (
    0.45 * (shortlist["total_medals"] / shortlist["total_medals"].max()) +
    0.35 * (shortlist["delta"].clip(lower=0) / max(shortlist["delta"].clip(lower=0).max(), 1)) +
    0.20 * (shortlist["medal_rate"] / max(shortlist["medal_rate"].max(), 1e-9))
)

shortlist = shortlist.sort_values("score_2028", ascending=False)

fig = px.bar(shortlist.head(15), x="NOC", y="score_2028", title="YPerf shortlist score for JO 2028")
fig.show()

shortlist.head(15)


,NOC,total_medals,delta,recent_avg,medal_rate,score_2028
0,USA,16774,278.370370,809.666667,1.0,0.823671
2,FRA,11971,255.202381,616.666667,1.0,0.680366
5,AUS,8379,348.820513,601.666667,1.0,0.642410
1,GBR,11998,146.845238,519.666667,1.0,0.613488
7,JPN,7721,326.047619,607.000000,1.0,0.610550
3,ITA,9351,231.785714,511.000000,1.0,0.595469
4,GER,8866,220.385965,593.333333,1.0,0.575346
75,ROC,561,561.000000,561.000000,1.0,0.565050
6,CAN,7907,235.730769,484.000000,1.0,0.559192
15,BRA,4601,321.166667,466.666667,1.0,0.523803


## Export des artefacts pour slides

Les tableaux sont exportes dans `reports/metrics/` pour insertion rapide dans le deck de soutenance.


In [12]:
OUT = Path("../reports/metrics")
OUT.mkdir(parents=True, exist_ok=True)

country_medals.head(20).to_csv(OUT / "eda02_top_countries.csv", index=False)
momentum.head(20).to_csv(OUT / "eda02_country_momentum.csv", index=False)
sport_medals.head(20).to_csv(OUT / "eda02_top_sports.csv", index=False)
sport_momentum.head(20).to_csv(OUT / "eda02_sport_momentum.csv", index=False)
country_eff.head(20).to_csv(OUT / "eda02_country_efficiency.csv", index=False)
shortlist.head(20).to_csv(OUT / "eda02_shortlist_2028.csv", index=False)

print(f"Exports written to: {OUT.resolve()}")


Exports written to: /Users/alx/Ynov/B3/Projet-Fil-Rouge-JO28/reports/metrics
